In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import json
import pandas as pd

filepath = "/content/drive/MyDrive/out_features.jsonl"

data = []
with open(filepath, "r", encoding="utf-8") as f:
    for line in f:
        try:
            data.append(json.loads(line))
        except json.JSONDecodeError:
            continue

df = pd.DataFrame(data)
print(f"Total rows loaded: {len(df)}")
print(f"Columns: {list(df.columns)}")
print()
print("Label distribution:")
print(df['label'].value_counts())

Total rows loaded: 80000
Columns: ['id', 'label', 'source', 'input', 'measurements', 'features']

Label distribution:
label
phishing    40000
benign      40000
Name: count, dtype: int64


In [3]:
print("Total rows:", len(df))
print("Total top-level columns:", len(df.columns))
print("\nTop-level columns:")
print(list(df.columns))

top_level_summary = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[col].dtype) for col in df.columns],
    "non_null_count": [df[col].notna().sum() for col in df.columns],
    "missing_count": [df[col].isna().sum() for col in df.columns],
    "missing_pct": [round(df[col].isna().mean() * 100, 2) for col in df.columns],
    "example_type": [
        df[col].dropna().map(lambda x: type(x).__name__).iloc[0]
        if len(df[col].dropna()) > 0 else None
        for col in df.columns
    ]
})

display(top_level_summary)

Total rows: 80000
Total top-level columns: 6

Top-level columns:
['id', 'label', 'source', 'input', 'measurements', 'features']


,column,dtype,non_null_count,missing_count,missing_pct,example_type
0,id,object,80000,0,0.0,str
1,label,object,80000,0,0.0,str
2,source,object,80000,0,0.0,str
3,input,object,80000,0,0.0,dict
4,measurements,object,80000,0,0.0,dict
5,features,object,80000,0,0.0,list


In [4]:
def collect_dict_keys(series):
    keys = set()
    for item in series.dropna():
        if isinstance(item, dict):
            keys.update(item.keys())
    return sorted(keys)

dict_columns = []

for col in df.columns:
    sample_values = df[col].dropna()
    if len(sample_values) > 0 and isinstance(sample_values.iloc[0], dict):
        dict_columns.append(col)

print("Dictionary-type columns:", dict_columns)

for col in dict_columns:
    keys = collect_dict_keys(df[col])
    print(f"\nColumn: {col}")
    print(f"Number of keys inside {col}: {len(keys)}")
    display(pd.DataFrame({"nested_key": keys}))

Dictionary-type columns: ['input', 'measurements']

Column: input
Number of keys inside input: 10


,nested_key
0,anchors
1,final_url
2,forms
3,iframes
4,meta
5,redirects
6,resources
7,title
8,url
9,visible_text



Column: measurements
Number of keys inside measurements: 22


,nested_key
0,anchor_count
1,contact_anchor_domains
2,credential_form_count
3,external_anchor_count
4,external_anchor_ratio
5,favicon_count
6,final_hostname
7,final_url_length
8,form_action_relationship_counts
9,form_count


In [7]:
measurements_df = pd.json_normalize(df["measurements"])

print("Number of measurement features:", measurements_df.shape[1])
print("\nMeasurement feature names:")
print(list(measurements_df.columns))

measurement_summary = pd.DataFrame({
    "measurement_feature": measurements_df.columns,
    "dtype": [str(measurements_df[col].dtype) for col in measurements_df.columns],
    "non_null_count": [measurements_df[col].notna().sum() for col in measurements_df.columns],
    "missing_count": [measurements_df[col].isna().sum() for col in measurements_df.columns],
    "missing_pct": [round(measurements_df[col].isna().mean() * 100, 2) for col in measurements_df.columns],
    "unique_values": [measurements_df[col].apply(lambda x: tuple(x) if isinstance(x, list) else x).nunique(dropna=True) for col in measurements_df.columns]
})

display(measurement_summary)

Number of measurement features: 27

Measurement feature names:
['url_length', 'final_url_length', 'final_hostname', 'hostname_label_count', 'redirect_count', 'form_count', 'credential_form_count', 'password_input_count', 'hidden_input_count', 'anchor_count', 'external_anchor_count', 'external_anchor_ratio', 'null_anchor_count', 'internal_nav_candidate_count', 'contact_anchor_domains', 'favicon_count', 'script_src_sample_count', 'stylesheet_href_sample_count', 'image_src_sample_count', 'visible_text_length', 'title_is_generic', 'form_action_relationship_counts.same_registrable_domain', 'form_action_relationship_counts.unknown', 'form_action_relationship_counts.unrelated_third_party', 'form_action_relationship_counts.known_identity_provider', 'form_action_relationship_counts.known_infrastructure_provider', 'form_action_relationship_counts.known_payment_or_app_store']


,measurement_feature,dtype,non_null_count,missing_count,missing_pct,unique_values
0,url_length,int64,80000,0,0.00,484
1,final_url_length,int64,80000,0,0.00,654
2,final_hostname,object,80000,0,0.00,32769
3,hostname_label_count,int64,80000,0,0.00,9
4,redirect_count,int64,80000,0,0.00,10
5,form_count,int64,80000,0,0.00,75
6,credential_form_count,int64,80000,0,0.00,20
7,password_input_count,int64,80000,0,0.00,18
8,hidden_input_count,int64,80000,0,0.00,87
9,anchor_count,int64,80000,0,0.00,101


In [8]:
feature_rows = []

for row_idx, row in df.iterrows():
    feature_list = row.get("features", [])

    if not isinstance(feature_list, list):
        continue

    for feature in feature_list:
        if isinstance(feature, dict):
            feature_rows.append({
                "row_index": row_idx,
                "label": row.get("label"),
                "feature_id": feature.get("id"),
                "direction": feature.get("direction"),
                "severity": feature.get("severity"),
                "statement": feature.get("statement"),
                "value": feature.get("value")
            })

features_long_df = pd.DataFrame(feature_rows)

print("Total feature occurrences:", len(features_long_df))
print("Number of unique feature IDs:", features_long_df["feature_id"].nunique())

display(features_long_df.head(20))

Total feature occurrences: 381464
Number of unique feature IDs: 36


,row_index,label,feature_id,direction,severity,statement,value
0,0,phishing,meta.robots_noindex_nofollow,suspicious,low,"Page sets robots directive 'NOINDEX, NOFOLLOW'.","{'content': 'NOINDEX, NOFOLLOW'}"
1,0,phishing,credential.password_input_present,suspicious,high,Page contains 1 password field(s).,{'count': 1}
2,0,phishing,credential.credential_terms_near_form,suspicious,medium,Form asks for credential-related information: ...,"{'terms': ['email', 'password', 'security', 's..."
3,0,phishing,form.empty_or_blank_action,suspicious,medium,Form uses an empty or blank action.,{'count': 1}
4,0,phishing,form.blank_action_js_submission_suspected,uncertain,low,Form has a blank action and may rely on JavaSc...,{'count': 1}
5,0,phishing,form.hidden_inputs,suspicious,low,Credential form includes hidden input field(s)...,"{'names': ['isUtf8', 'source'], 'count': 2}"
6,0,phishing,brand.title_domain_mismatch,suspicious,medium,"Page title claims 'Outlook Web App', but the r...","{'title': 'Outlook Web App', 'domain': 'swiftw..."
7,0,phishing,link.null_or_void_anchors,suspicious,low,Page includes 1 null or void link(s).,"{'count': 1, 'total': 1, 'examples': ['<empty>']}"
8,0,phishing,login.missing_recovery_or_help_flow,suspicious,low,"Login form lacks normal recovery, help, or acc...","{'checked_terms': ['contact', 'create account'..."
9,1,phishing,credential.password_input_present,suspicious,high,Page contains 1 password field(s).,{'count': 1}


In [9]:
feature_id_summary = (
    features_long_df
    .groupby("feature_id")
    .agg(
        total_count=("feature_id", "count"),
        phishing_count=("label", lambda x: (x == "phishing").sum()),
        benign_count=("label", lambda x: (x == "benign").sum()),
        directions=("direction", lambda x: sorted(x.dropna().unique())),
        severities=("severity", lambda x: sorted(x.dropna().unique())),
        example_statement=("statement", "first")
    )
    .reset_index()
    .sort_values("total_count", ascending=False)
)

print("Number of unique heuristic features:", len(feature_id_summary))

display(feature_id_summary)

Number of unique heuristic features: 36


,feature_id,total_count,phishing_count,benign_count,directions,severities,example_statement
20,link.null_or_void_anchors,45457,22392,23065,[suspicious],[low],Page includes 1 null or void link(s).
23,navigation.functional_internal_links,43987,8196,35791,[benign],[low],Page has functional internal navigation links.
30,support.contact_domain_match,38051,13628,24423,[benign],[medium],Support or contact information matches the pag...
9,credential.credential_terms_near_form,32940,19862,13078,[suspicious],[medium],Form asks for credential-related information: ...
24,page.generic_login_without_brand_claim,27015,17551,9464,[uncertain],[medium],Page contains a generic login form without a c...
17,iframe.hidden_iframe,24056,5649,18407,[suspicious],[low],Page contains a hidden iframe.
10,credential.password_input_present,19597,13562,6035,[suspicious],[high],Page contains 1 password field(s).
15,form.hidden_inputs,19313,10241,9072,[suspicious],[low],Credential form includes hidden input field(s)...
21,login.missing_recovery_or_help_flow,18903,13684,5219,[suspicious],[low],"Login form lacks normal recovery, help, or acc..."
11,form.action_same_org_domain,18016,10074,7942,[benign],[low],Form submission stays within the same organiza...


In [10]:
print("==== Feature Count Summary ====")
print("Top-level columns:", len(df.columns))
print("Measurement features:", measurements_df.shape[1])
print("Unique heuristic feature IDs:", features_long_df["feature_id"].nunique())
print("Total heuristic feature occurrences:", len(features_long_df))
print()

print("Rows:", len(df))
print("Labels:")
print(df["label"].value_counts())

==== Feature Count Summary ====
Top-level columns: 6
Measurement features: 27
Unique heuristic feature IDs: 36
Total heuristic feature occurrences: 381464

Rows: 80000
Labels:
label
phishing    40000
benign      40000
Name: count, dtype: int64


In [11]:
# List all features in the dataset

# 1) Measurement features
measurement_features = list(pd.json_normalize(df["measurements"]).columns)

# 2) Heuristic feature IDs inside the "features" list
heuristic_features = sorted({
    feature.get("id")
    for feature_list in df["features"].dropna()
    if isinstance(feature_list, list)
    for feature in feature_list
    if isinstance(feature, dict) and feature.get("id") is not None
})

print("Measurement features:")
print(f"Count: {len(measurement_features)}")
for f in measurement_features:
    print("-", f)

print("\nHeuristic features:")
print(f"Count: {len(heuristic_features)}")
for f in heuristic_features:
    print("-", f)

print("\nTotal features:")
print(len(measurement_features) + len(heuristic_features))

Measurement features:
Count: 27
- url_length
- final_url_length
- final_hostname
- hostname_label_count
- redirect_count
- form_count
- credential_form_count
- password_input_count
- hidden_input_count
- anchor_count
- external_anchor_count
- external_anchor_ratio
- null_anchor_count
- internal_nav_candidate_count
- contact_anchor_domains
- favicon_count
- script_src_sample_count
- stylesheet_href_sample_count
- image_src_sample_count
- visible_text_length
- title_is_generic
- form_action_relationship_counts.same_registrable_domain
- form_action_relationship_counts.unknown
- form_action_relationship_counts.unrelated_third_party
- form_action_relationship_counts.known_identity_provider
- form_action_relationship_counts.known_infrastructure_provider
- form_action_relationship_counts.known_payment_or_app_store

Heuristic features:
Count: 36
- brand.domain_lookalike
- brand.favicon_domain_mismatch
- brand.title_domain_match_strong
- brand.title_domain_mismatch
- contact.identity_domain_mis

In [16]:
import pandas as pd
import numpy as np

# Flatten measurements
measurements_df = pd.json_normalize(df["measurements"])
measurements_df["label"] = df["label"]
measurements_df["id"] = df["id"]

# Extract heuristic features into long format
feature_rows = []

for i, row in df.iterrows():
    feature_list = row["features"]

    if isinstance(feature_list, list):
        for feature in feature_list:
            if isinstance(feature, dict):
                feature_rows.append({
                    "id": row["id"],
                    "label": row["label"],
                    "feature_id": feature.get("id"),
                    "direction": feature.get("direction"),
                    "severity": feature.get("severity"),
                    "statement": feature.get("statement"),
                    "value": feature.get("value")
                })

features_long_df = pd.DataFrame(feature_rows)

print("Measurements table shape:", measurements_df.shape)
print("Heuristic features table shape:", features_long_df.shape)

display(measurements_df.head())
display(features_long_df.head())

Measurements table shape: (80000, 29)
Heuristic features table shape: (381464, 7)


,url_length,final_url_length,final_hostname,hostname_label_count,redirect_count,form_count,credential_form_count,password_input_count,hidden_input_count,anchor_count,...,visible_text_length,title_is_generic,form_action_relationship_counts.same_registrable_domain,form_action_relationship_counts.unknown,form_action_relationship_counts.unrelated_third_party,form_action_relationship_counts.known_identity_provider,form_action_relationship_counts.known_infrastructure_provider,form_action_relationship_counts.known_payment_or_app_store,label,id
0,63,63,app46657.swiftway.in,3,0,1,1,1,2,1,...,257,False,NaN,NaN,NaN,NaN,NaN,NaN,phishing,6911d21446483f1ce32438ed
1,36,36,quitshadow.com,2,0,1,1,1,0,2,...,201,False,1.0,NaN,NaN,NaN,NaN,NaN,phishing,6911d21546483f1ce32438ee
2,28,36,quitshadow.com,2,4,1,1,1,0,2,...,201,False,1.0,NaN,NaN,NaN,NaN,NaN,phishing,6911d21946483f1ce32438ef
3,56,56,streijl.eu,2,0,2,1,1,0,12,...,3417,False,NaN,NaN,NaN,NaN,NaN,NaN,phishing,6911d21d46483f1ce32438f1
4,83,83,bafkreiaww32bm77n2jnyv3w6v4ncnd7koyalrexjty22j...,4,0,1,1,1,2,0,...,131,False,NaN,NaN,NaN,NaN,NaN,NaN,phishing,6911d23712b7a8e079909292


,id,label,feature_id,direction,severity,statement,value
0,6911d21446483f1ce32438ed,phishing,meta.robots_noindex_nofollow,suspicious,low,"Page sets robots directive 'NOINDEX, NOFOLLOW'.","{'content': 'NOINDEX, NOFOLLOW'}"
1,6911d21446483f1ce32438ed,phishing,credential.password_input_present,suspicious,high,Page contains 1 password field(s).,{'count': 1}
2,6911d21446483f1ce32438ed,phishing,credential.credential_terms_near_form,suspicious,medium,Form asks for credential-related information: ...,"{'terms': ['email', 'password', 'security', 's..."
3,6911d21446483f1ce32438ed,phishing,form.empty_or_blank_action,suspicious,medium,Form uses an empty or blank action.,{'count': 1}
4,6911d21446483f1ce32438ed,phishing,form.blank_action_js_submission_suspected,uncertain,low,Form has a blank action and may rely on JavaSc...,{'count': 1}


In [17]:
measurement_features = [c for c in measurements_df.columns if c not in ["id", "label"]]

print("Number of measurement features:", len(measurement_features))

for feature in measurement_features:
    print("-", feature)

Number of measurement features: 27
- url_length
- final_url_length
- final_hostname
- hostname_label_count
- redirect_count
- form_count
- credential_form_count
- password_input_count
- hidden_input_count
- anchor_count
- external_anchor_count
- external_anchor_ratio
- null_anchor_count
- internal_nav_candidate_count
- contact_anchor_domains
- favicon_count
- script_src_sample_count
- stylesheet_href_sample_count
- image_src_sample_count
- visible_text_length
- title_is_generic
- form_action_relationship_counts.same_registrable_domain
- form_action_relationship_counts.unknown
- form_action_relationship_counts.unrelated_third_party
- form_action_relationship_counts.known_identity_provider
- form_action_relationship_counts.known_infrastructure_provider
- form_action_relationship_counts.known_payment_or_app_store


In [18]:
heuristic_features = sorted(features_long_df["feature_id"].dropna().unique())

print("Number of heuristic features:", len(heuristic_features))

for feature in heuristic_features:
    print("-", feature)

Number of heuristic features: 36
- brand.domain_lookalike
- brand.favicon_domain_mismatch
- brand.title_domain_match_strong
- brand.title_domain_mismatch
- contact.identity_domain_mismatch
- content.coercive_urgency_near_form
- content.copyright_domain_mismatch
- content.file_lure_terms
- content.non_credential_transactional_page
- credential.credential_terms_near_form
- credential.password_input_present
- form.action_same_org_domain
- form.blank_action_js_submission_suspected
- form.empty_or_blank_action
- form.external_action
- form.hidden_inputs
- identity.provider_expected_domain
- iframe.hidden_iframe
- link.brand_text_domain_mismatch
- link.high_external_anchor_ratio
- link.null_or_void_anchors
- login.missing_recovery_or_help_flow
- meta.robots_noindex_nofollow
- navigation.functional_internal_links
- page.generic_login_without_brand_claim
- page.interstitial_without_credential_collection
- page.low_semantic_content
- page.rendering_incomplete_or_script_dependent
- redirect.cros

In [19]:
heuristic_summary = (
    features_long_df
    .groupby(["feature_id", "label"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# Make sure both columns exist
if "phishing" not in heuristic_summary.columns:
    heuristic_summary["phishing"] = 0

if "benign" not in heuristic_summary.columns:
    heuristic_summary["benign"] = 0

heuristic_summary["total"] = heuristic_summary["phishing"] + heuristic_summary["benign"]
heuristic_summary["phishing_ratio"] = heuristic_summary["phishing"] / heuristic_summary["total"]
heuristic_summary["benign_ratio"] = heuristic_summary["benign"] / heuristic_summary["total"]

heuristic_summary = heuristic_summary.sort_values("total", ascending=False)

display(heuristic_summary)

label,feature_id,benign,phishing,total,phishing_ratio,benign_ratio
20,link.null_or_void_anchors,23065,22392,45457,0.492597,0.507403
23,navigation.functional_internal_links,35791,8196,43987,0.186328,0.813672
30,support.contact_domain_match,24423,13628,38051,0.358151,0.641849
9,credential.credential_terms_near_form,13078,19862,32940,0.602975,0.397025
24,page.generic_login_without_brand_claim,9464,17551,27015,0.649676,0.350324
17,iframe.hidden_iframe,18407,5649,24056,0.234827,0.765173
10,credential.password_input_present,6035,13562,19597,0.692045,0.307955
15,form.hidden_inputs,9072,10241,19313,0.530265,0.469735
21,login.missing_recovery_or_help_flow,5219,13684,18903,0.723906,0.276094
11,form.action_same_org_domain,7942,10074,18016,0.559170,0.440830


In [20]:
phishing_features = heuristic_summary.sort_values(
    ["phishing_ratio", "phishing"],
    ascending=False
)

display(phishing_features.head(30))

label,feature_id,benign,phishing,total,phishing_ratio,benign_ratio
34,url.http_not_https,21,1056,1077,0.980501,0.019499
32,url.email_identifier,3,44,47,0.936170,0.063830
19,link.high_external_anchor_ratio,95,1248,1343,0.929263,0.070737
3,brand.title_domain_mismatch,557,6155,6712,0.917014,0.082986
6,content.copyright_domain_mismatch,350,1615,1965,0.821883,0.178117
1,brand.favicon_domain_mismatch,200,696,896,0.776786,0.223214
29,redirect.multi_hop,825,2705,3530,0.766289,0.233711
0,brand.domain_lookalike,237,732,969,0.755418,0.244582
5,content.coercive_urgency_near_form,203,622,825,0.753939,0.246061
21,login.missing_recovery_or_help_flow,5219,13684,18903,0.723906,0.276094


In [21]:
benign_features = heuristic_summary.sort_values(
    ["benign_ratio", "benign"],
    ascending=False
)

display(benign_features.head(30))

label,feature_id,benign,phishing,total,phishing_ratio,benign_ratio
18,link.brand_text_domain_mismatch,7915,1236,9151,0.135067,0.864933
2,brand.title_domain_match_strong,502,98,600,0.163333,0.836667
23,navigation.functional_internal_links,35791,8196,43987,0.186328,0.813672
17,iframe.hidden_iframe,18407,5649,24056,0.234827,0.765173
7,content.file_lure_terms,724,325,1049,0.309819,0.690181
35,url.long_url,2174,1039,3213,0.323374,0.676626
14,form.external_action,1127,614,1741,0.352671,0.647329
30,support.contact_domain_match,24423,13628,38051,0.358151,0.641849
8,content.non_credential_transactional_page,1251,700,1951,0.358790,0.641210
4,contact.identity_domain_mismatch,5607,3917,9524,0.411277,0.588723


In [22]:
severity_summary = (
    features_long_df
    .groupby(["label", "severity"])
    .size()
    .reset_index(name="count")
    .sort_values(["label", "count"], ascending=[True, False])
)

display(severity_summary)

,label,severity,count
1,benign,low,122397
2,benign,medium,63185
0,benign,high,8336
4,phishing,low,92021
5,phishing,medium,79427
3,phishing,high,16098


In [23]:
direction_summary = (
    features_long_df
    .groupby(["label", "direction"])
    .size()
    .reset_index(name="count")
    .sort_values(["label", "count"], ascending=[True, False])
)

display(direction_summary)

,label,direction,count
2,benign,suspicious,106555
0,benign,benign,68781
3,benign,uncertain,18465
1,benign,neutral,117
6,phishing,suspicious,127055
4,phishing,benign,32094
7,phishing,uncertain,28175
5,phishing,neutral,222


In [29]:
numeric_measurements = measurements_df[measurement_features].select_dtypes(include=["int64", "float64"]).columns

measurement_stats = (
    measurements_df
    .groupby("label")[numeric_measurements]
    .agg(["mean", "median", "std", "min", "max"])
)

display(measurement_stats)

url_length                             final_url_length         \
               mean median        std min   max             mean median   
label                                                                     
benign    55.521425   42.0  59.047792  13  1930        62.549775   43.0   
phishing  53.090525   43.0  38.052814  15  1103        54.169800   44.0   

                               ...  \
                std min   max  ...   
label                          ...   
benign    91.384385  13  2000  ...   
phishing  45.324806  15  1103  ...   

         form_action_relationship_counts.known_infrastructure_provider         \
                                                                  mean median   
label                                                                           
benign                                                 15.0              15.0   
phishing                                                NaN               NaN   

                               \
               std  min   max   
label                           
benign    19.79899  1.0  29.0   
phishing       NaN  NaN   NaN   

         form_action_relationship_counts.known_payment_or_app_store         \
                                                               mean median   
label                                                                        
benign                                                  1.0            1.0   
phishing                                                NaN            NaN   

                         
          std  min  max  
label                    
benign    0.0  1.0  1.0  
phishing  NaN  NaN  NaN  

[2 rows x 120 columns]

In [30]:
comparison_rows = []

for col in numeric_measurements:
    phishing_values = measurements_df[measurements_df["label"] == "phishing"][col]
    benign_values = measurements_df[measurements_df["label"] == "benign"][col]

    comparison_rows.append({
        "feature": col,
        "phishing_mean": phishing_values.mean(),
        "benign_mean": benign_values.mean(),
        "difference": phishing_values.mean() - benign_values.mean(),
        "abs_difference": abs(phishing_values.mean() - benign_values.mean())
    })

measurement_comparison = pd.DataFrame(comparison_rows)
measurement_comparison = measurement_comparison.sort_values("abs_difference", ascending=False)

display(measurement_comparison)

,feature,phishing_mean,benign_mean,difference,abs_difference
17,visible_text_length,1216.551325,3064.372525,-1847.821200,1847.821200
8,anchor_count,14.971450,70.953825,-55.982375,55.982375
12,internal_nav_candidate_count,6.545950,50.601325,-44.055375,44.055375
16,image_src_sample_count,12.511825,24.369300,-11.857475,11.857475
1,final_url_length,54.169800,62.549775,-8.379975,8.379975
14,script_src_sample_count,11.180125,17.546975,-6.366850,6.366850
9,external_anchor_count,2.905875,8.944125,-6.038250,6.038250
0,url_length,53.090525,55.521425,-2.430900,2.430900
15,stylesheet_href_sample_count,5.183725,7.438275,-2.254550,2.254550
13,favicon_count,1.064250,2.785550,-1.721300,1.721300


In [31]:
missing_rows = []

for col in measurement_features:
    missing_rows.append({
        "feature": col,
        "missing_count": measurements_df[col].isna().sum(),
        "missing_pct": measurements_df[col].isna().mean() * 100
    })

missing_summary = pd.DataFrame(missing_rows)
missing_summary = missing_summary.sort_values("missing_pct", ascending=False)

display(missing_summary)

,feature,missing_count,missing_pct
26,form_action_relationship_counts.known_payment_...,79998,99.99750
25,form_action_relationship_counts.known_infrastr...,79998,99.99750
24,form_action_relationship_counts.known_identity...,79981,99.97625
22,form_action_relationship_counts.unknown,79826,99.78250
23,form_action_relationship_counts.unrelated_thir...,78307,97.88375
21,form_action_relationship_counts.same_registrab...,61984,77.48000
4,redirect_count,0,0.00000
3,hostname_label_count,0,0.00000
0,url_length,0,0.00000
1,final_url_length,0,0.00000


In [32]:
outlier_rows = []

for col in numeric_measurements:
    q1 = measurements_df[col].quantile(0.25)
    q3 = measurements_df[col].quantile(0.75)
    iqr = q3 - q1

    lower_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr

    outlier_count = (
        (measurements_df[col] < lower_limit) |
        (measurements_df[col] > upper_limit)
    ).sum()

    outlier_rows.append({
        "feature": col,
        "outlier_count": outlier_count,
        "outlier_pct": outlier_count / len(measurements_df) * 100,
        "lower_limit": lower_limit,
        "upper_limit": upper_limit
    })

outlier_summary = pd.DataFrame(outlier_rows)
outlier_summary = outlier_summary.sort_values("outlier_pct", ascending=False)

display(outlier_summary)

,feature,outlier_count,outlier_pct,lower_limit,upper_limit
6,password_input_count,19597,24.49625,0.00000,0.00000
7,hidden_input_count,19351,24.18875,0.00000,0.00000
10,external_anchor_ratio,10716,13.39500,-0.21195,0.35325
15,stylesheet_href_sample_count,10163,12.70375,-8.00000,16.00000
4,form_count,9718,12.14750,-1.50000,2.50000
9,external_anchor_count,8873,11.09125,-9.00000,15.00000
16,image_src_sample_count,6499,8.12375,-35.00000,61.00000
11,null_anchor_count,5892,7.36500,-9.00000,15.00000
14,script_src_sample_count,5870,7.33750,-22.00000,42.00000
13,favicon_count,5459,6.82375,-3.00000,5.00000


In [33]:
feature_name = "url_length"

q1 = measurements_df[feature_name].quantile(0.25)
q3 = measurements_df[feature_name].quantile(0.75)
iqr = q3 - q1

lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr

outlier_examples = measurements_df[
    (measurements_df[feature_name] < lower_limit) |
    (measurements_df[feature_name] > upper_limit)
]

print("Feature:", feature_name)
print("Outlier count:", len(outlier_examples))

display(outlier_examples[["id", "label", feature_name]].head(30))

Feature: url_length
Outlier count: 3852


,id,label,url_length
10,6911d24712b7a8e079909298,phishing,204
72,6911d32f12b7a8e0799092da,phishing,159
83,6911d36312b7a8e0799092eb,phishing,1093
226,6911d5b112b7a8e0799093b1,phishing,117
289,6911d6b812b7a8e0799093fd,phishing,125
374,6911d97212b7a8e07990948b,phishing,118
398,6911da2412b7a8e0799094b7,phishing,143
401,6911da3112b7a8e0799094bd,phishing,141
435,6911db6012b7a8e0799094f4,phishing,413
449,6911db9212b7a8e079909505,phishing,129


In [34]:
corr = measurements_df[numeric_measurements].corr().abs()

similar_pairs = []

for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        feature_1 = corr.columns[i]
        feature_2 = corr.columns[j]
        correlation = corr.iloc[i, j]

        if correlation >= 0.90:
            similar_pairs.append({
                "feature_1": feature_1,
                "feature_2": feature_2,
                "correlation": correlation
            })

similar_features_df = pd.DataFrame(similar_pairs)
similar_features_df = similar_features_df.sort_values("correlation", ascending=False)

display(similar_features_df)

,feature_1,feature_2,correlation
12,external_anchor_ratio,form_action_relationship_counts.known_infrastr...,1.000000
0,url_length,form_action_relationship_counts.known_infrastr...,1.000000
2,hostname_label_count,form_action_relationship_counts.known_infrastr...,1.000000
1,final_url_length,form_action_relationship_counts.known_infrastr...,1.000000
3,form_count,form_action_relationship_counts.known_infrastr...,1.000000
5,credential_form_count,form_action_relationship_counts.known_infrastr...,1.000000
10,anchor_count,form_action_relationship_counts.known_infrastr...,1.000000
7,password_input_count,form_action_relationship_counts.known_infrastr...,1.000000
11,external_anchor_count,form_action_relationship_counts.known_infrastr...,1.000000
14,internal_nav_candidate_count,form_action_relationship_counts.known_infrastr...,1.000000


In [35]:
audit_rows = []

for col in numeric_measurements:
    phishing_mean = measurements_df[measurements_df["label"] == "phishing"][col].mean()
    benign_mean = measurements_df[measurements_df["label"] == "benign"][col].mean()
    missing_pct = measurements_df[col].isna().mean() * 100

    audit_rows.append({
        "feature": col,
        "type": "measurement",
        "missing_pct": missing_pct,
        "phishing_mean": phishing_mean,
        "benign_mean": benign_mean,
        "difference": phishing_mean - benign_mean,
        "abs_difference": abs(phishing_mean - benign_mean),
        "note": ""
    })

feature_audit_df = pd.DataFrame(audit_rows)
feature_audit_df = feature_audit_df.sort_values("abs_difference", ascending=False)

display(feature_audit_df)

,feature,type,missing_pct,phishing_mean,benign_mean,difference,abs_difference,note
17,visible_text_length,measurement,0.00000,1216.551325,3064.372525,-1847.821200,1847.821200,
8,anchor_count,measurement,0.00000,14.971450,70.953825,-55.982375,55.982375,
12,internal_nav_candidate_count,measurement,0.00000,6.545950,50.601325,-44.055375,44.055375,
16,image_src_sample_count,measurement,0.00000,12.511825,24.369300,-11.857475,11.857475,
1,final_url_length,measurement,0.00000,54.169800,62.549775,-8.379975,8.379975,
14,script_src_sample_count,measurement,0.00000,11.180125,17.546975,-6.366850,6.366850,
9,external_anchor_count,measurement,0.00000,2.905875,8.944125,-6.038250,6.038250,
0,url_length,measurement,0.00000,53.090525,55.521425,-2.430900,2.430900,
15,stylesheet_href_sample_count,measurement,0.00000,5.183725,7.438275,-2.254550,2.254550,
13,favicon_count,measurement,0.00000,1.064250,2.785550,-1.721300,1.721300,


In [36]:
output_folder = "/content/drive/MyDrive/"

measurement_comparison.to_csv(output_folder + "measurement_comparison.csv", index=False)
heuristic_summary.to_csv(output_folder + "heuristic_feature_summary.csv", index=False)
outlier_summary.to_csv(output_folder + "outlier_summary.csv", index=False)
similar_features_df.to_csv(output_folder + "similar_measurement_features.csv", index=False)
feature_audit_df.to_csv(output_folder + "feature_audit.csv", index=False)

print("Saved analysis files to Google Drive.")

Saved analysis files to Google Drive.
